## Standardized pipeline for creating oligos

Change the variable and path names accordingly in the first section

Issues:
- Need to change so standardize automatically removes everything past * and outputs that it did so for original .csv

In [59]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../tools")

import generate_TCRAFT_oligopool as gt
import pandas as pd

from config import ALL_DATA_DIR

import random    # stdlib; this is the RNG TCRAFT's codon sampler draws from
import numpy as np    # pandas .sample() draws from numpy's global RNG

In [60]:
# load csvs
AS_df = pd.read_csv(ALL_DATA_DIR / "AS_Top_96_TCRs.csv")
Ctrl_df = pd.read_csv(ALL_DATA_DIR / "Positive_Ctrl_TCRs.csv")

In [61]:
### ONLY CHANGE VALUES OF THESE CELLS ###

# note the run ID 
run_ID = "01"

# directory suffix
dir_suffix = "PCtrl_pilot"

# change the csv ID
oligo_csv_name = "pilot"

# how many oligos to select from input
# if want to select all oligos, use len()
selected_oligos = len(Ctrl_df)

# load df
df_to_use = Ctrl_df

# change this if a TCR repeatedly fails to generate, since a fixed seed
# reproduces a bad draw exactly instead of escaping it on a rerun
SEED = 7702

In [62]:
# create OUTPUT DIR name
OUTPUT_DIR = ALL_DATA_DIR / str(f"{run_ID}_{dir_suffix}")

In [63]:
# set seed
def set_seed(seed=SEED):
    """Reset both global RNGs the pipeline draws from.

    Called at the top of each pipeline step so that step is reproducible
    regardless of what else has been run in this kernel.
    """
    random.seed(seed)    # TCRAFT codon sampling (Generate.py:203)
    np.random.seed(seed)    # pandas df.sample in standardize_to_sample_format

In [64]:
# standardize 
set_seed()
gt.standardize_to_sample_format(
    df=df_to_use,
    select=selected_oligos,
    output_dir=OUTPUT_DIR
)

,Name,V_alpha,V_beta,CDR3_alpha,CDR3_beta,J_alpha,J_beta
0,gp100,TRAV21,TRBV7-3,CAVLSSGGSNYKLTF,CASSFIGGTDTQYF,TRAJ53,TRBJ2-3
1,MART-1,TRAV12-2,TRBV6-4,CAVNFGGGKLIF,CASSLSFGTEAFF,TRAJ23,TRBJ1-1
2,gp100,TRAV17,TRBV19,CATDGDTPLVF,CASSIGGPYEQYF,TRAJ29,TRBJ2-7


In [65]:
set_seed()
gt.generate_oligos(
    tcraft_csv=OUTPUT_DIR / "standardized_TCRs.csv",
    prefix=run_ID,
    name=oligo_csv_name,
    data_dir=OUTPUT_DIR
)

*******************
* TCRAFT-generate *
*******************

Generating CDR3 oligos...


100%|██████████| 3/3 [00:00<00:00, 12.80it/s]

Generated 3 valid CDR3 oligos.
0/3 oligos (0.0%) are over 300 bp long.
Saving output files to /Users/bellwu/Py_programming/TCR_oligos/all_data/01_PCtrl_pilot/01_oligos_pilot


PosixPath('/Users/bellwu/Py_programming/TCR_oligos/all_data/01_PCtrl_pilot/01_oligos_pilot/All_3_CDR3_oligos.csv')

In [66]:
# record the seed alongside the outputs so a pool order on file can be
# traced back to the run that produced it
import json

run_params = {
    "run_ID": run_ID,
    "seed": SEED,
    "n_selected": int(selected_oligos),
    "input_csv": "AS_Top_96_TCRs.csv",
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "numpy": np.__version__,
}
with open(OUTPUT_DIR / f"{run_ID}_run_params.json", "w") as f:
    json.dump(run_params, f, indent=2)

run_params

{'run_ID': '01',
 'seed': 7702,
 'n_selected': 3,
 'input_csv': 'AS_Top_96_TCRs.csv',
 'python': '3.12.14',
 'pandas': '3.0.5',
 'numpy': '2.5.3'}

In [67]:
gt.validate_oligos(
    oligos_csv=OUTPUT_DIR/str(f"{run_ID}_oligos_{oligo_csv_name}")/str(f"All_{selected_oligos}_CDR3_oligos.csv"),
    prefix=run_ID,
    name="test_run",
    data_dir=OUTPUT_DIR
)

*******************
* TCRAFT-validate *
*******************

Beginning assembly validation...


100%|██████████| 3/3 [00:00<00:00, 524.03it/s]

Completed assembly validation.
3 out of 3 TCRs correctly assembled.
Saving output files to /Users/bellwu/Py_programming/TCR_oligos/all_data/01_PCtrl_pilot/01_validation_test_run


PosixPath('/Users/bellwu/Py_programming/TCR_oligos/all_data/01_PCtrl_pilot/01_validation_test_run')

In [68]:
gt.select_sequences_by_pool(
    oligos_csv=OUTPUT_DIR/str(f"{run_ID}_oligos_{oligo_csv_name}")/str(f"All_{selected_oligos}_CDR3_oligos.csv"),
    prefix=run_ID,
    data_dir=OUTPUT_DIR
)

{'all': PosixPath('/Users/bellwu/Py_programming/TCR_oligos/all_data/01_PCtrl_pilot/01_IDT_oligos.csv'),
 'A': PosixPath('/Users/bellwu/Py_programming/TCR_oligos/all_data/01_PCtrl_pilot/01_PoolA_IDT_oligos.csv'),
 'B': PosixPath('/Users/bellwu/Py_programming/TCR_oligos/all_data/01_PCtrl_pilot/01_PoolB_IDT_oligos.csv')}

In [69]:
print("Script successfuly run")

Script successfuly run
